# Leave-One-State-Out Prediction

Each of the 31 states (6 in D1, 5 in D2 and 20 in D3) is held out once. A model is fitted on the
other states of its dataset and predicts the spectrum of the held-out state. The measured values
of the held-out state enter no fit. The models are K-K GP-ECM, ECM (circuit), GP-RBF-0, GP-RBF-c,
K-K GP-0 and qGP-DRT. With the output files in `experiments/` a run takes a few seconds.
Computing all fits again takes about 40 min with `N_JOBS = 3`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

from eis import data, datasets, env, fitting, loso, qgpdrt, utils
from eis.gpexp import VARIANTS

RECOMPUTE = False   # True computes every output file again
N_JOBS = 3          # number of worker processes for the fits
# folder of the output files (experiments/ in the repository root)
OUT = os.path.join(env.DIR_BASE, "experiments")
os.makedirs(OUT, exist_ok=True)

# PyTorch warns that a tensor made from a read-only NumPy array must not be written. The models
# only read these tensors. PYTHONWARNINGS passes the same filter to the worker processes.
warnings.filterwarnings("ignore", message="The given NumPy array is not writable")
os.environ["PYTHONWARNINGS"] = "ignore:The given NumPy array is not writable:UserWarning"

DATA = {name: data.load_dataset(name) for name in datasets.DATASETS}
CASES = utils.held_out_cases(DATA)

## K-K GP-ECM and ECM (Circuit)

K-K GP-ECM models the spectrum on the grid $\mathcal{G}$ of all $G$ points $x = (\omega, v)$ of a
dataset. It is a Gaussian process (GP) whose mean $\mathbf{m} = (m_\mathrm{re}, m_\mathrm{im})$
is the equivalent circuit ECM-W:

$$Z_\mathrm{ECM}(\omega;\theta) = j\omega L + R_e + Z_\mathrm{Z}(R_\mathrm{SEI}, Q_\mathrm{SEI},
\alpha_\mathrm{SEI}) + Z_\mathrm{Z}(R_\mathrm{ct}, Q_\mathrm{dl}, \alpha_\mathrm{dl})
+ Z_\mathrm{W}(A_w, \tau),$$

with $Z_\mathrm{Z}(R, Q, \alpha) = R/(1 + RQ(j\omega)^\alpha)$ and
$Z_\mathrm{W}(A_w, \tau) = (A_w/\sqrt{j\omega})\coth\sqrt{j\omega\tau}$. The ten parameters
$\theta$ of each training state minimize
$\sum_i \lvert Z_\mathrm{ECM}(\omega_i;\theta) - Z(\omega_i)\rvert^2$ over its frequencies. The
held-out state receives $\theta$ interpolated across the training states at its $v$.

The real residual $\mathbf{u} = \mathrm{Re}\,Z - m_\mathrm{re}$ on $\mathcal{G}$ has the prior
$\mathcal{N}(\mathbf{0}, \mathbf{K})$. The Gram matrix $\mathbf{K}$ comes from the RBF kernel
$k(x, x') = \sigma_f^2 \exp\big(-\sum_d (x_d - x'_d)^2 / (2\ell_d^2)\big)$ on
$(\log\omega, v)$. The imaginary residual is $\mathbf{H}_\mathcal{G}\mathbf{u}$, where
$\mathbf{H}_\mathcal{G} = \mathbf{I}_{n_v} \otimes \mathbf{H}$. The matrix $\mathbf{H}$ is the
discrete Hilbert transform on the frequency grid of one state (`eis.hilbert.hilbert_matrix`).
The posterior-mean prediction $\hat{Z}$ therefore satisfies
$\mathrm{Im}\,\hat{Z} - m_\mathrm{im} = \mathbf{H}_\mathcal{G}(\mathrm{Re}\,\hat{Z} - m_\mathrm{re})$.
ECM (circuit) is the interpolated circuit alone.

The GP hyperparameters $(\sigma_f, \ell, \sigma_{n,\mathrm{re}}, \sigma_{n,\mathrm{im}})$ start at
$\log\sigma_f = 0$, $\log\ell_d = 0$ and $\log\sigma_{n,\mathrm{re}} = \log\sigma_{n,\mathrm{im}} = -2$.
Adam (300 steps, rate 0.05) maximizes the marginal likelihood of the training points. Then L-BFGS
(5 calls of at most 100 iterations) continues the maximization. The circuit fit of one spectrum
starts from values read off the spectrum. It uses Adam (500 steps, rate 0.05) and then L-BFGS
(5 calls of at most 200 iterations). The interpolation of $\theta$ is linear in $v$. It acts on
the logarithm of each positive parameter and on the logit of each exponent $\alpha$. Outside the
range of the training states the parameters stay at the value of the nearest training state.

For one held-out state, `eis.loso.fit_kkgp_ecm` fits K-K GP-ECM (`KKGP(kernel="rbf", mean="M1")`
of `eis.gp`, method label `KKGP-ecmmean`). It also scores ECM (circuit) in the columns
`ecm_rmse_re` and `ecm_rmse_im`.

In [2]:
def compute_kkgp_ecm():
    cases = [(name, s, circuit, df) for name, s, df in CASES for circuit in fitting.CIRCUITS]
    rows = utils.fit_parallel(loso.fit_kkgp_ecm, cases, N_JOBS)
    return rows.sort_values(["dataset", "circuit", "held_out_v"]).reset_index(drop=True)


kkgp_ecm = utils.load_or_compute(os.path.join(OUT, "02_kkgp_ecm.csv"), compute_kkgp_ecm, RECOMPUTE)

[2026-10-19 05:38:13][WAR] KKGP LBFGS step aborted (_LinAlgError); keeping best params.


31 fits in 11.1 min


## GP-RBF-0 and GP-RBF-c

The unconstrained baselines model $\mathrm{Re}\,Z$ and $\mathrm{Im}\,Z$ as two independent GPs on
$(\log\omega, v)$. Each channel has its own RBF kernel and its own noise level. GP-RBF-0 has the
mean zero, and GP-RBF-c has one fitted constant mean per channel. Since the two GPs are
independent, a prediction need not satisfy the K-K relation. Both use the GP settings of K-K
GP-ECM and are fitted by `eis.loso.fit_gp_rbf` (`GPRBF(mean="zero")` and `GPRBF(mean="const")`
of `eis.gp`).

In [3]:
def compute_gp_rbf():
    cases = [(name, s, label, mean, df) for name, s, df in CASES
             for label, mean in VARIANTS.items()]
    rows = utils.fit_parallel(loso.fit_gp_rbf, cases, N_JOBS)
    return rows.sort_values(["dataset", "method", "held_out_v"]).reset_index(drop=True)


gp_rbf = utils.load_or_compute(os.path.join(OUT, "02_gp_rbf.csv"), compute_gp_rbf, RECOMPUTE)

62 fits in 3.5 min


## K-K GP-0

K-K GP-0 is the constrained GP of K-K GP-ECM with the mean $\mathbf{m} = \mathbf{0}$, so its
prediction satisfies $\mathrm{Im}\,\hat{Z} = \mathbf{H}_\mathcal{G}\,\mathrm{Re}\,\hat{Z}$. It is
`KKGP(kernel="rbf", mean="M2")` with the method label `KKGP-zeromean` (`eis.loso.fit_kkgp_zero`).

In [4]:
def compute_kkgp_zero():
    rows = utils.fit_parallel(loso.fit_kkgp_zero, CASES, N_JOBS)
    return rows.sort_values(["dataset", "method", "held_out_v"]).reset_index(drop=True)


kkgp_zero = utils.load_or_compute(
    os.path.join(OUT, "02_kkgp_zero.csv"), compute_kkgp_zero, RECOMPUTE)

31 fits in 8.7 min


## qGP-DRT

The model qGP-DRT (Maradesa, Py and Ciucci, 2024) is a GP over the distribution of relaxation
times $\gamma$ of several states. The model of state $s$ is

$$Z_s(\omega) = R_s + j\omega L_s + \int \frac{\gamma_s(\ln\tau)}{1 + j\omega\tau}\,\mathrm{d}\ln\tau,$$

discretized on $T$ values of $\tau$. The version in `eis/qgpdrt.py` adapts it to
leave-one-state-out. It adds the inductance $L_s$ and the lower bound $10^{-2}$ on the
standardized noise level $\sigma_n$. The held-out state is one block of the latent vector with no
observation rows. The Hamiltonian Monte Carlo (HMC) sampler draws 5000 samples of this block and
drops the first 1000. The predictive mean at each frequency is the mean over the other 4000
samples. Each held-out state has one fixed HMC seed (`HMC_SEEDS` in `eis/qgpdrt.py`).

The number of $\tau$ values is $T$ = round(10 times the number of frequency decades). It is at
most 30 on D3 and at most 80 on D1 and D2. The hyperparameters come from 3 L-BFGS-B starts. The
files of the authors of qGP-DRT are in `eis/quasigp/` without changes. The qGP-DRT fits run with
one BLAS thread per worker process.

In [5]:
# one BLAS thread per worker process
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(name, "1")


def compute_qgpdrt():
    cases = [(name, s, variant, 5000) for name, s, variant in qgpdrt.build_cases(list(DATA))]
    rows = utils.fit_parallel(qgpdrt.fit_qgpdrt, cases, N_JOBS)
    return rows.sort_values(["dataset", "method", "held_out_v"]).reset_index(drop=True)


qgp = utils.load_or_compute(os.path.join(OUT, "02_qgpdrt.csv"), compute_qgpdrt, RECOMPUTE)

31 fits in 16.5 min


## RMSE by Dataset and Region

The RMSE of one channel at a held-out state is
$\big(\frac{1}{n_\omega}\sum_i (y_i - \hat{y}_i)^2\big)^{1/2}$ over its $n_\omega$ frequencies.
Here $\hat{y}$ is the predictive mean. The table averages the RMSE of $\mathrm{Re}\,Z$ and of
$\mathrm{Im}\,Z$ (ohm) over the $n$ states of each region. A boundary state has the smallest or
the largest $v$ of its dataset, and every other state is interior. The values have two
significant digits.

In [6]:
def region_means(df, re="rmse_re", im="rmse_im"):
    m = df.groupby(["dataset", "region"])[[re, im]].mean().loc[utils.REGIONS]
    return m.set_axis(["Re", "Im"], axis=1)


ecm = kkgp_ecm[kkgp_ecm["circuit"] == "ECM-W"]
table = pd.concat({
    "K-K GP-ECM": region_means(ecm),
    "GP-RBF-c": region_means(gp_rbf[gp_rbf["method"] == "GP-RBF-c"]),
    "ECM (circuit)": region_means(ecm, "ecm_rmse_re", "ecm_rmse_im"),
    "qGP-DRT": region_means(qgp[qgp["method"] == "qGP-DRT"]),
}, axis=1).map(utils.two_digits)
table.insert(0, ("", "n"), ecm.groupby(["dataset", "region"]).size().loc[utils.REGIONS])
table.index = [f"{d.split('_')[0]} {r}" for d, r in utils.REGIONS]
table

K-K GP-ECM          GP-RBF-c         ECM (circuit)           \
              n         Re       Im       Re      Im            Re       Im   
D1 interior   4      0.042    0.026    0.038   0.021         0.043    0.029   
D1 boundary   2     0.0078   0.0081    0.084   0.020         0.014    0.013   
D2 interior   3       0.33    0.051   0.0099  0.0052          0.33    0.051   
D2 boundary   2     0.0093   0.0065    0.017  0.0077        0.0039   0.0023   
D3 interior  18     0.0025  0.00044  0.00095  0.0060        0.0023  0.00062   
D3 boundary   2      0.014    0.042    0.020   0.073         0.013    0.042   

            qGP-DRT          
                 Re      Im  
D1 interior   0.035   0.032  
D1 boundary   0.086   0.029  
D2 interior   0.049  0.0084  
D2 boundary    0.37   0.062  
D3 interior  0.0033  0.0013  
D3 boundary    0.52    0.11

## Selected States and the GP Correction

GP-RBF-0 and K-K GP-0 are compared at the D3 state $v = 3.2261$. At this state GP-RBF-0 has the
largest share of sign errors (notebook 03). For K-K GP-ECM on D3, the RMSE of the boundary states
is set against that of the interior states. The real RMSE at the three interior states of D2 is
listed for K-K GP-ECM and ECM (circuit). For K-K GP-ECM, the largest of these values (at
$v = 3.69$) is divided by the mean $\lvert Z\rvert$ of D2. The size of the GP correction to the
circuit mean is $\max_i \lvert\hat{y}_i - m_i\rvert$ over the held-out frequencies of a channel.
The columns `gp_corr_re` and `gp_corr_im` hold this size.

In [7]:
s = 3.2261
pair = pd.DataFrame({
    "GP-RBF-0": gp_rbf[(gp_rbf["method"] == "GP-RBF-0")
                       & np.isclose(gp_rbf["held_out_v_nom"], s)].iloc[0],
    "K-K GP-0": kkgp_zero[np.isclose(kkgp_zero["held_out_v_nom"], s)].iloc[0],
}).T[["rmse_re", "rmse_im"]].astype(float)
print(f"D3, state v = {s} held out (RMSE in ohm):")
print(pair.to_string(formatters={"rmse_re": "{:.4f}".format, "rmse_im": "{:.3f}".format}))
print(f"Imaginary RMSE of GP-RBF-0 / K-K GP-0: "
      f"{pair.loc['GP-RBF-0', 'rmse_im'] / pair.loc['K-K GP-0', 'rmse_im']:.1f}")

reg = ecm.groupby(["dataset", "region"])[["rmse_re", "rmse_im"]].mean().loc["D3_mustafa"]
ratio = reg.loc["boundary"] / reg.loc["interior"]
print(f"\nD3, K-K GP-ECM, boundary RMSE / interior RMSE: "
      f"Re {ratio['rmse_re']:.1f}, Im {ratio['rmse_im']:.1f}")

d2 = ecm[(ecm["dataset"] == "D2_rashid") & (ecm["region"] == "interior")]
print("\nD2 interior states, real RMSE (ohm):")
print(d2[["held_out_v", "rmse_re", "ecm_rmse_re"]]
      .rename(columns={"held_out_v": "state v", "rmse_re": "K-K GP-ECM",
                       "ecm_rmse_re": "ECM (circuit)"})
      .to_string(index=False, float_format="{:.3f}".format))
mean_abs_z = float(np.hypot(DATA["D2_rashid"]["zr"], DATA["D2_rashid"]["zj"]).mean())
worst = float(d2.loc[np.isclose(d2["held_out_v_nom"], 3.69), "rmse_re"].iloc[0])
print(f"Real RMSE at v = 3.69 / mean |Z| of D2: {worst / mean_abs_z:.1f}")

corr = np.concatenate([ecm["gp_corr_re"], ecm["gp_corr_im"]])
print(f"\nGP correction to the circuit mean over the 31 states and both channels (ohm): "
      f"smallest {corr.min():.1e}, median {np.median(corr):.1e}, largest {corr.max():.1e}")

D3, state v = 3.2261 held out (RMSE in ohm):
         rmse_re rmse_im
GP-RBF-0  0.0005   0.170
K-K GP-0  0.0005   0.012
Imaginary RMSE of GP-RBF-0 / K-K GP-0: 14.4

D3, K-K GP-ECM, boundary RMSE / interior RMSE: Re 5.5, Im 96.5

D2 interior states, real RMSE (ohm):
 state v  K-K GP-ECM  ECM (circuit)
   3.590       0.010          0.010
   3.690       0.978          0.978
   3.850       0.006          0.005
Real RMSE at v = 3.69 / mean |Z| of D2: 31.1

GP correction to the circuit mean over the 31 states and both channels (ohm): smallest 7.7e-04, median 2.5e-03, largest 3.8e-02
